# Progetto finale del corso
## Problema A: Inquinamento atmosferico

In [1]:
import numpy as np 
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt

import json

from pathlib import Path
from shapely.geometry import Point
from shapely.geometry import shape

# c'è anche da importare sklearn

In [10]:
data_path = Path('../data/raw')

files = {'grid':'trentino-grid.geojson',
         'adm_reg':'administrative_regions_Trentino.json',
         'weather':'meteotrentino-weather-station-data.json',
         'precip':'precipitation-trentino.csv',
         'precip-avail':'precipitation-trentino-data-availability.csv',
         'SET-1':'SET-nov-2013.csv',
         'SET-2':'SET-dec-2013.csv',
         'SET-lines':'line.csv',
         'twitter':'social-pulse-trentino.geojson'}

df_grid = gpd.read_file(data_path / files['grid'])

DataSourceError: ../data/raw/trentino-grid.geojson: No such file or directory

In [ ]:
df_grid.plot('cellId') 

In [ ]:
'''   Mancano dati e comunque è inutile )= (
with open(data_path / files['adm_reg']) as f:
    adm_reg_json = json.load(f)

adm_reg = gpd.GeoDataFrame(adm_reg_json['items'])

# convert GeoJSON dicts to shapely geometries
adm_reg['geometry'] = adm_reg['geometry'].apply(shape)

# set geometry column
adm_reg = adm_reg.set_geometry('geometry')
for i, geom in enumerate(adm_reg['geometry']):
    try:
        shape(geom)
    except Exception as e:
        print(f"Row {i}: {e}")
        print(geom)
        break


In [ ]:
with open(data_path / files['weather']) as f:
    weather_json = json.load(f)

weather = gpd.GeoDataFrame(weather_json['features'])
# weather[weather['station'] == 'T0071'][['precipitations.0000', 'precipitations.0015', 'precipitations.0030', 'precipitations.0045']]
weather

In [ ]:
weather['geometry'] = weather['geomPoint.geom'].apply(lambda x:Point(x['coordinates'][0], x['coordinates'][1]))
weather.drop(columns=['geomPoint.geom'],inplace=True)
weather

In [ ]:
# elevazione è solo una caratteristica della stazione?
print(weather['elevation'])

In [ ]:
time_mask = (weather['timestamp'] == 1383260400)
ax = df_grid.plot()
weather[time_mask].plot(column='temperatures.0000',ax=ax)

In [ ]:
station_mask = (weather['station'] == 'T0071')

plt.plot('timestamp', 'minTemperature', data=weather[station_mask])
plt.plot('timestamp', 'maxTemperature', data=weather[station_mask])

In [ ]:
APPA_df = pd.read_csv('./data_trentino/external/APPA_inquinamento_aria_Nov_Dec_2013.csv', encoding_errors="ignore")

APPA_df

In [ ]:
winds_cols = [c for c in weather.columns if c.startswith("winds.")]

new_cols = []

for c in winds_cols:

    split_cols = weather[c].str.split("@", n=1, expand=True)

    # ensure two columns always exist
    split_cols = split_cols.reindex(columns=[0, 1])

    split_cols.columns = [f"{c}_spd", f"{c}_dir"]

    split_cols = split_cols.apply(pd.to_numeric, errors="coerce")

    new_cols.append(split_cols)

weather = pd.concat([weather] + new_cols, axis=1)




In [ ]:
weather.drop(columns=winds_cols,inplace=True)
weather

In [ ]:
weather.columns[weather.columns.duplicated()]

In [ ]:
meteo_df = None

weather_df = weather

for station_id in weather_df['station'].drop_duplicates():

    df_station = weather_df[weather_df["station"] == station_id]

    # === COLONNE PRECIPITAZIONI ===
    temp_cols = [c for c in df_station.columns if c.startswith("temperatures.")]
    prec_cols = [c for c in df_station.columns if c.startswith("precipitations.")]
    winds_cols_spd = [c for c in df_station.columns if (c.startswith("winds.") and c.endswith("_spd"))]
    winds_cols_dir = [c for c in df_station.columns if (c.startswith("winds.") and c.endswith("_dir"))]

    # lista finale
    rows = []
    prec = []
    winds_spd = []
    winds_dir = []

    # === COSTRUZIONE NUOVO DATAFRAME ===
    for _, row in df_station.iterrows():

        date = pd.to_datetime(row["date"])

        for i in range(len(temp_cols)):

            # estrae HHMM
            hhmm = temp_cols[i].split(".")[1]

            hour = hhmm[:2]
            minute = hhmm[2:]

            # costruzione datetime
            dt = date.replace(
                hour=int(hour),
                minute=int(minute)
            )

            # formato richiesto:
            # minuto-ora-giorno-mese-anno
            # esempio 1215140313
            custom_time = dt.strftime("%M%H%d%m%y")

            rows.append({
                'datetime': custom_time,
                'temperatures_' + station_id: row[temp_cols[i]],
                'precipitations_' + station_id: row[prec_cols[i]],
                'winds_spd_' + station_id: row[winds_cols_spd[i]],
                'winds_dir_' + station_id: row[winds_cols_dir[i]]
            }) 
    
    df = pd.DataFrame(rows)

    if meteo_df is None:
        meteo_df = df
    else:
        meteo_df = meteo_df.merge(df, on="datetime")



meteo_df